## 02 — Fetch & Match News

Loads the raw news dump, keyword-matches each headline to Nifty50 tickers,
explodes multi-ticker rows, then hands off to the same build-content/insert
steps as before.

In [1]:
import pandas as pd
import ast

df = pd.read_csv(r"C:\Users\acer\Desktop\nifty50-v2\backend\notebooks\data\stock_news_2016 to 2026.csv")
df = df[df['date'] >= '2015-01-01'].copy().reset_index(drop=True)

print(f"Shape after 2015 filter : {df.shape}")
print(f"Columns : {df.columns.tolist()}")

Shape after 2015 filter : (139919, 10)
Columns : ['date', 'title', 'description', 'url', 'source_file', 'categories', 'matched_keywords', 'relevance_score', 'has_negation', 'impact_tier']


In [2]:
TICKER_KEYWORDS = {
    "RELIANCE.NS": [
        "reliance", "reliance industries", "reliance retail",
        "reliance group", "mukesh ambani", "jio", "ril"
    ],
    "TCS.NS": [
        "tcs", "tata consultancy"
    ],
    "HDFCBANK.NS": [
        "hdfc", "hdfc bank", "hdfcbank", "hdfc merger",
        "sashidhar jagdishan"
    ],
    "INFY.NS": [
        "infosys", "infy", "narayana murthy",
        "salil parekh", "nandan nilekani"
    ],
    "ICICIBANK.NS": [
        "icici", "icici bank", "icicibank", "sandeep bakhshi"
    ],
    "HINDUNILVR.NS": [
        "hindustan unilever", "hul", "surf excel",
        "lifebuoy", "dove", "lux soap", "rin detergent"
    ],
    "ITC.NS": [
        "itc", "itc limited", "itc cigarette", "itc fmcg",
        "itc hotel", "wills", "sanjiv puri"
    ],
    "SBIN.NS": [
        "sbi", "state bank", "state bank of india",
        "sbi bank", "sbi npa", "dinesh kumar khara"
    ],
    "BHARTIARTL.NS": [
        "airtel", "bharti airtel", "bharti",
        "sunil mittal", "gopal vittal"
    ],
    "KOTAKBANK.NS": [
        "kotak", "kotak bank", "kotak mahindra",
        "uday kotak", "ashok vaswani"
    ],
    "LT.NS": [
        "larsen", "l&t", "larsen toubro",
        "larsen and toubro", "sn subrahmanyan"
    ],
    "AXISBANK.NS": [
        "axis", "axis bank", "axisbank", "amitabh chaudhry"
    ],
    "ASIANPAINT.NS": [
        "asian paints", "asian paint", "amit syngle",
        "nerolac", "berger paints"
    ],
    "MARUTI.NS": [
        "maruti", "maruti suzuki", "suzuki india",
        "swift dzire", "maruti brezza", "maruti ertiga",
        "maruti alto", "rc bhargava", "hisashi takeuchi"
    ],
    "SUNPHARMA.NS": [
        "sun pharma", "sun pharmaceutical",
        "dilip shanghvi", "ranbaxy"
    ],
    "TITAN.NS": [
        "titan", "titan company", "tanishq",
        "fastrack", "caratlane", "titan watches"
    ],
    "BAJFINANCE.NS": [
        "bajaj finance", "rajeev jain"
    ],
    "NESTLEIND.NS": [
        "nestle", "nestle india", "maggi",
        "nescafe", "kitkat", "munch"
    ],
    "WIPRO.NS": [
        "wipro", "rishad premji"
    ],
    "ULTRACEMCO.NS": [
        "ultratech", "ultratech cement", "kk maheshwari"
    ],
    "POWERGRID.NS": [
        "power grid", "pgcil", "powergrid"
    ],
    "NTPC.NS": [
        "ntpc", "national thermal power", "gurdeep singh"
    ],
    "TECHM.NS": [
        "tech mahindra", "techm", "cp gurnani", "mohit joshi"
    ],
    "HCLTECH.NS": [
        "hcl", "hcl tech", "hcl technologies",
        "hcltech", "c vijayakumar"
    ],
    "BAJAJFINSV.NS": [
        "bajaj finserv", "finserv", "bajaj allianz", "sanjiv bajaj"
    ],
    "ONGC.NS": [
        "ongc", "oil and natural gas", "arun kumar singh"
    ],
    "JSWSTEEL.NS": [
        "jsw", "jsw steel", "sajjan jindal"
    ],
    "TMPV.NS": [
        "tata motors", "jaguar land rover", "jlr",
        "range rover", "tata nexon", "tata punch",
        "tata safari", "tata ev", "shailesh chandra"
    ],
    "ADANIENT.NS": [
        "adani", "adani enterprises", "gautam adani",
        "adani group", "hindenburg adani", "adani bribery",
        "adani airports", "adani wilmar", "adanient"
    ],
    "COALINDIA.NS": [
        "coal india", "coalindia"
    ],
    "GRASIM.NS": [
        "grasim", "grasim industries", "birla paints"
    ],
    "DIVISLAB.NS": [
        "divi", "divi labs", "divis labs",
        "divi laboratories", "murali divi"
    ],
    "DRREDDY.NS": [
        "dr reddy", "dr reddys", "drreddy", "erez israeli"
    ],
    "EICHERMOT.NS": [
        "eicher", "royal enfield", "bullet 350",
        "re himalayan", "siddhartha lal"
    ],
    "CIPLA.NS": [
        "cipla", "umang vohra"
    ],
    "BPCL.NS": [
        "bpcl", "bharat petroleum", "bpcl refinery"
    ],
    "HEROMOTOCO.NS": [
        "hero motocorp", "hero moto", "hero honda",
        "hero splendor", "pawan munjal", "niranjan gupta"
    ],
    "BRITANNIA.NS": [
        "britannia", "britannia industries",
        "good day biscuit", "varun berry", "rajneet kohli"
    ],
    "APOLLOHOSP.NS": [
        "apollo", "apollo hospitals", "apollo hospital",
        "apollo pharmacy", "prathap reddy"
    ],
    "TATACONSUM.NS": [
        "tata consumer", "tetley", "tata tea",
        "tata salt", "tata starbucks", "sunil d'souza"
    ],
    "HINDALCO.NS": [
        "hindalco", "hindalco industries",
        "novelis", "satish pai", "kumar mangalam birla"
    ],
    "TATASTEEL.NS": [
        "tata steel", "tv narendran"
    ],
    "UPL.NS": [
        "upl", "upl limited", "advanta seeds",
        "upl agro", "jaidev shroff"
    ],
    "SHREECEM.NS": [
        "shree cement", "shreecem", "benu gopal bangur"
    ],
    "SBILIFE.NS": [
        "sbi life", "sbi life insurance"
    ],
    "BAJAJ-AUTO.NS": [
        "bajaj auto", "bajaj pulsar", "bajaj dominar",
        "rajiv bajaj", "bajaj export"
    ],
    "HDFCLIFE.NS": [
        "hdfc life", "hdfc life insurance", "vibha padalkar"
    ],
    "INDUSINDBK.NS": [
        "indusind", "indusind bank", "sumant kathpalia"
    ],
    "M&M.NS": [
        "mahindra", "m&m", "mahindra and mahindra",
        "anand mahindra", "mahindra thar", "mahindra xuv",
        "mahindra scorpio", "mahindra ev", "rajesh jejurikar"
    ],
    "ADANIPORTS.NS": [
        "adani ports", "apsez", "mundra port",
        "adani logistics", "karan adani"
    ],
}

In [3]:
def find_tickers(text):
    text = str(text).lower()
    match = []
    for ticker in TICKER_KEYWORDS:
        for word in TICKER_KEYWORDS[ticker]:
            if word in text:
                match.append(ticker)
                break
    return match

TEXT_COL = df["title"].fillna("").astype(str) + " " + df["description"].fillna("").astype(str)
df["ticker"] = TEXT_COL.apply(find_tickers)

matched = df[df["ticker"].str.len() > 0].copy()
print(f"Matched: {len(matched):,}")

# explode so each ticker gets its own row
df_exploded = matched.explode("ticker").copy()
df_exploded = df_exploded.rename(columns={"source_file": "source"})
df_exploded = df_exploded.drop_duplicates(subset=["title", "ticker"]).reset_index(drop=True)

print(f"After explode + dedup : {len(df_exploded):,}")
print(df_exploded["ticker"].value_counts().to_string())

Matched: 39,661
After explode + dedup : 51,523
ticker
RELIANCE.NS      6689
SBIN.NS          4379
HDFCBANK.NS      2955
ICICIBANK.NS     2522
ITC.NS           2184
INFY.NS          1973
BHARTIARTL.NS    1954
ADANIENT.NS      1917
TCS.NS           1852
TMPV.NS          1807
M&M.NS           1668
MARUTI.NS        1533
AXISBANK.NS      1402
HINDUNILVR.NS    1109
KOTAKBANK.NS     1034
DIVISLAB.NS      1011
TATASTEEL.NS      976
WIPRO.NS          917
LT.NS             725
ONGC.NS           700
HEROMOTOCO.NS     685
TITAN.NS          670
HCLTECH.NS        652
BAJAJ-AUTO.NS     608
INDUSINDBK.NS     598
SUNPHARMA.NS      591
JSWSTEEL.NS       573
NTPC.NS           547
EICHERMOT.NS      540
COALINDIA.NS      518
BAJFINANCE.NS     462
BPCL.NS           461
DRREDDY.NS        435
CIPLA.NS          435
TECHM.NS          428
HINDALCO.NS       417
ADANIPORTS.NS     396
ASIANPAINT.NS     393
NESTLEIND.NS      378
BRITANNIA.NS      334
ULTRACEMCO.NS     298
APOLLOHOSP.NS     266
HDFCLIFE.NS       241


In [4]:
# rename ticker that changed (Tata Motors passenger vehicles spin-off)
df['ticker'] = df['ticker'].replace('TATAMOTORS.NS', 'TMPV.NS')

def parse_ticker(t):
    if isinstance(t, list):
        return t
    try:
        parsed = ast.literal_eval(str(t))
        if isinstance(parsed, list):
            return parsed
        return [str(t)]
    except Exception:
        return [str(t)]

df['ticker_list'] = df['ticker'].apply(parse_ticker)
df_exploded = df.explode('ticker_list').copy()
df_exploded['ticker'] = df_exploded['ticker_list']
df_exploded = df_exploded.drop(columns=['ticker_list'])
df_exploded = df_exploded.drop_duplicates(subset=['title', 'ticker'])
df_exploded = df_exploded.reset_index(drop=True)

print(f"After explode + dedup : {len(df_exploded):,}")
print("\nRows per ticker:")
print(df_exploded['ticker'].value_counts().to_string())

After explode + dedup : 151,779

Rows per ticker:
ticker
RELIANCE.NS      6689
SBIN.NS          4379
HDFCBANK.NS      2955
ICICIBANK.NS     2522
ITC.NS           2184
INFY.NS          1973
BHARTIARTL.NS    1954
ADANIENT.NS      1917
TCS.NS           1852
TMPV.NS          1807
M&M.NS           1668
MARUTI.NS        1533
AXISBANK.NS      1402
HINDUNILVR.NS    1109
KOTAKBANK.NS     1034
DIVISLAB.NS      1011
TATASTEEL.NS      976
WIPRO.NS          917
LT.NS             725
ONGC.NS           700
HEROMOTOCO.NS     685
TITAN.NS          670
HCLTECH.NS        652
BAJAJ-AUTO.NS     608
INDUSINDBK.NS     598
SUNPHARMA.NS      591
JSWSTEEL.NS       573
NTPC.NS           547
EICHERMOT.NS      540
COALINDIA.NS      518
BAJFINANCE.NS     462
BPCL.NS           461
DRREDDY.NS        435
CIPLA.NS          435
TECHM.NS          428
HINDALCO.NS       417
ADANIPORTS.NS     396
ASIANPAINT.NS     393
NESTLEIND.NS      378
BRITANNIA.NS      334
ULTRACEMCO.NS     298
APOLLOHOSP.NS     266
HDFCLIFE.NS       2

## Build article content (title + description)

In [5]:
def build_content(row):
    title = str(row.get('title', '')).strip()
    desc = str(row.get('description', '')).strip()
    if desc and desc != title and desc != 'nan':
        return f"{title}. {desc}"
    return title

df_exploded['content'] = df_exploded.apply(build_content, axis=1)

print("Sample content:")
for c in df_exploded['content'].head(3):
    print(f"  {c[:100]}")

Sample content:
  Lending to foreign step-down arms of Indian firms gets costlier. RBI slaps 2% additional provision f
  IDBI Bank to raise Rs 900 cr through Basel-III compliant bonds. Many banks have been raising money t
  Forex reserves up $943 mn. India's foreign exchange reserves rose $943 million to about $352 billion


## Insert into news_articles

Sentiment columns are left NULL here on purpose — scoring happens in `03_score_sentiment_and_combine.ipynb`, so this stage only handles raw article ingestion. Safe to re-run: if `news_articles` already has rows for a ticker, that ticker's insert is skipped.

In [6]:
import sys
sys.path.append('..')
from utils.db import get_connection

conn = get_connection()
cur = conn.cursor()
cur.execute("TRUNCATE TABLE news_articles RESTART IDENTITY;")
conn.commit()
cur.close()
conn.close()
print("news_articles cleared.")

news_articles cleared.


In [7]:
import sys
sys.path.append('..')
from datetime import datetime
from urllib.parse import urlparse
from psycopg2.extras import execute_values
from utils.db import get_connection

def get_source(url):
    try:
        if not url or url == 'nan' or url == 'URL_NOT_AVAILABLE':
            return 'unknown'
        return urlparse(str(url)).netloc.replace('www.', '')
    except Exception:
        return 'unknown'

rows_to_insert = []
for _, row in df_exploded.iterrows():
    ticker = str(row['ticker']).strip()

    try:
        published_at = datetime.strptime(str(row['date']), "%Y-%m-%d")
    except Exception:
        published_at = None

    url = str(row.get('url', '')).strip()
    content = str(row.get('content', '')).strip()
    if not content or content == 'nan':
        continue

    rows_to_insert.append((
        ticker, content, get_source(url), published_at,
        url if url not in ['nan', 'URL_NOT_AVAILABLE', ''] else None
    ))

print(f"Prepared {len(rows_to_insert):,} rows — inserting in batches...")

conn = get_connection()
cur = conn.cursor()

BATCH_SIZE = 2000
for i in range(0, len(rows_to_insert), BATCH_SIZE):
    batch = rows_to_insert[i:i + BATCH_SIZE]
    execute_values(cur, """
        INSERT INTO news_articles (ticker, content, source, published_at, url)
        VALUES %s
    """, batch)
    conn.commit()
    print(f"  Inserted {min(i + BATCH_SIZE, len(rows_to_insert)):,} / {len(rows_to_insert):,}")

cur.close()
conn.close()
print(f"\nTotal inserted : {len(rows_to_insert):,}")

Prepared 151,779 rows — inserting in batches...
  Inserted 2,000 / 151,779
  Inserted 4,000 / 151,779
  Inserted 6,000 / 151,779
  Inserted 8,000 / 151,779
  Inserted 10,000 / 151,779
  Inserted 12,000 / 151,779
  Inserted 14,000 / 151,779
  Inserted 16,000 / 151,779
  Inserted 18,000 / 151,779
  Inserted 20,000 / 151,779
  Inserted 22,000 / 151,779
  Inserted 24,000 / 151,779
  Inserted 26,000 / 151,779
  Inserted 28,000 / 151,779
  Inserted 30,000 / 151,779
  Inserted 32,000 / 151,779
  Inserted 34,000 / 151,779
  Inserted 36,000 / 151,779
  Inserted 38,000 / 151,779
  Inserted 40,000 / 151,779
  Inserted 42,000 / 151,779
  Inserted 44,000 / 151,779
  Inserted 46,000 / 151,779
  Inserted 48,000 / 151,779
  Inserted 50,000 / 151,779
  Inserted 52,000 / 151,779
  Inserted 54,000 / 151,779
  Inserted 56,000 / 151,779
  Inserted 58,000 / 151,779
  Inserted 60,000 / 151,779
  Inserted 62,000 / 151,779
  Inserted 64,000 / 151,779
  Inserted 66,000 / 151,779
  Inserted 68,000 / 151,779
  In

## Verify

In [8]:
conn = get_connection()
counts = pd.read_sql("""
    SELECT ticker, COUNT(*) AS articles
    FROM news_articles
    GROUP BY ticker
    ORDER BY articles DESC
""", conn)
conn.close()

print(f"Total rows    : {counts['articles'].sum():,}")
print(f"Total tickers : {len(counts)}")
print(counts.to_string())

Total rows    : 151,779
Total tickers : 51
           ticker  articles
0             nan    100256
1     RELIANCE.NS      6689
2         SBIN.NS      4379
3     HDFCBANK.NS      2955
4    ICICIBANK.NS      2522
5          ITC.NS      2184
6         INFY.NS      1973
7   BHARTIARTL.NS      1954
8     ADANIENT.NS      1917
9          TCS.NS      1852
10        TMPV.NS      1807
11         M&M.NS      1668
12      MARUTI.NS      1533
13    AXISBANK.NS      1402
14  HINDUNILVR.NS      1109
15   KOTAKBANK.NS      1034
16    DIVISLAB.NS      1011
17   TATASTEEL.NS       976
18       WIPRO.NS       917
19          LT.NS       725
20        ONGC.NS       700
21  HEROMOTOCO.NS       685
22       TITAN.NS       670
23     HCLTECH.NS       652
24  BAJAJ-AUTO.NS       608
25  INDUSINDBK.NS       598
26   SUNPHARMA.NS       591
27    JSWSTEEL.NS       573
28        NTPC.NS       547
29   EICHERMOT.NS       540
30   COALINDIA.NS       518
31  BAJFINANCE.NS       462
32        BPCL.NS       461
33   

C:\Users\acer\AppData\Local\Temp\ipykernel_22888\3225479853.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  counts = pd.read_sql("""


In [9]:
import sys
sys.path.append('..')
from utils.db import get_connection

conn = get_connection()
cur = conn.cursor()

cur.execute("""
    DELETE FROM news_articles
    WHERE ticker IS NULL
       OR ticker = 'nan'
       OR TRIM(ticker) = ''
""")
deleted = cur.rowcount
conn.commit()
cur.close()
conn.close()

print(f"Deleted {deleted:,} rows with invalid ticker.")

Deleted 100,256 rows with invalid ticker.


In [10]:
conn = get_connection()
counts = pd.read_sql("""
    SELECT ticker, COUNT(*) AS articles
    FROM news_articles
    GROUP BY ticker
    ORDER BY articles DESC
""", conn)
conn.close()

print(f"Total rows    : {counts['articles'].sum():,}")
print(f"Total tickers : {len(counts)}")
print(counts.to_string())

C:\Users\acer\AppData\Local\Temp\ipykernel_22888\3225479853.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  counts = pd.read_sql("""


Total rows    : 51,523
Total tickers : 50
           ticker  articles
0     RELIANCE.NS      6689
1         SBIN.NS      4379
2     HDFCBANK.NS      2955
3    ICICIBANK.NS      2522
4          ITC.NS      2184
5         INFY.NS      1973
6   BHARTIARTL.NS      1954
7     ADANIENT.NS      1917
8          TCS.NS      1852
9         TMPV.NS      1807
10         M&M.NS      1668
11      MARUTI.NS      1533
12    AXISBANK.NS      1402
13  HINDUNILVR.NS      1109
14   KOTAKBANK.NS      1034
15    DIVISLAB.NS      1011
16   TATASTEEL.NS       976
17       WIPRO.NS       917
18          LT.NS       725
19        ONGC.NS       700
20  HEROMOTOCO.NS       685
21       TITAN.NS       670
22     HCLTECH.NS       652
23  BAJAJ-AUTO.NS       608
24  INDUSINDBK.NS       598
25   SUNPHARMA.NS       591
26    JSWSTEEL.NS       573
27        NTPC.NS       547
28   EICHERMOT.NS       540
29   COALINDIA.NS       518
30  BAJFINANCE.NS       462
31        BPCL.NS       461
32       CIPLA.NS       435
33    